In [ ]:
# Imports
import numpy as np
import pandas as pd
import xarray as xr
from minisom import MiniSom
import matplotlib as mpl
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from matplotlib import colormaps
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.feature as cf
from itertools import product
import itertools
from sankeyflow import Sankey
from sammon import sammon
import igraph as ig
import hexMinisom

In [ ]:
dataset = xr.open_dataarray('data/Z500FiltAnoms_ERA5_v3.nc')

latSlice = slice(20, 80) #20N, 80N
lonSlice = slice(200, 310) #160W, 50W
dataarray = dataset.sel(lat=latSlice, lon=lonSlice)
dataarray = dataarray.stack(latlon=['lat', 'lon']).values

print(dataarray.shape)

In [ ]:
def som_corr(som):
    weights = som.get_weights()
    neurons = list(product(range(weights.shape[0]), range(weights.shape[1])))
    wm = {neuron: [] for neuron in neurons}
    avgs = {neuron: [] for neuron in neurons}
    
    # sort the data into their winning neurons
    for i, x in enumerate(dataarray):
        wm[som.winner(x)].append(i)
        
    # calculate the average of each neuron
    for i in range(len(neurons)):
        avgs[neurons[i]] = dataset.sel(lat=latSlice, lon=lonSlice)[wm[neurons[i]]].mean(dim='time', skipna=True).stack(latlon=['lat', 'lon']).values
        
    # find each days correlation with respect to the average of its neuron
    averagesList = []
    for i in dataarray:
        averagesList.append(avgs[som.winner(i)])
        
    corr = []
    for i in range(len(dataarray)):
        corr.append(np.corrcoef(dataarray[i], averagesList[i])[0, 1])
    
    return sum(corr) / len(corr)

def pathway_error(som, data):
    errorCount = 0
    prevNode = som.winner(data[0])
    
    for day in data[1:]:
        curNode = som.winner(day)
        # node with a manhattan distance <= 1 will be adjacent or the same node
        manhattanDist = abs(prevNode[0]-curNode[0]) + abs(prevNode[1]-curNode[1])
        
        # calculate the nonadjacent hexagons
        if manhattanDist > 1:
            # account for parity in the hexagon grid
            if prevNode[1] % 2 == 0:
                if curNode != (prevNode[0] - 1, prevNode[0] + 1) and curNode != (prevNode[0] - 1, prevNode[0] - 1):
                    errorCount += 1
                    
            else:
                if curNode != (prevNode[0] + 1, prevNode[0] + 1) and curNode != (prevNode[0] + 1, prevNode[0] - 1):
                    errorCount += 1
            
        # update the prevNode         
        prevNode = curNode
          
    # return the percentage of errors          
    return errorCount / (data.shape[0] - 1)

In [ ]:
def train_som(rows, cols, sigma, learning_rate, decay_function, 
              neighborhood_function, topology, seed, iters):
    
    som = MiniSom(rows, cols, dataarray.shape[1], sigma=sigma, learning_rate=learning_rate, 
                     neighborhood_function=neighborhood_function, decay_function=decay_function, 
                     random_seed=seed, topology=topology)
    som.random_weights_init(dataarray)
    som.train(dataarray, iters, True)
    
    return som

def hex_plot(x_neurons, y_neurons, projection=None):
    """Create a matplot lib figure with an axis for each neuron already positioned into the hexagonal shape"""
    
    # create figure
    totRows = x_neurons * 3
    totCols = y_neurons * 2
    fig = plt.figure(figsize=[totRows, totCols])
    axs = {}
    
    for x, y in list(product(range(x_neurons), range(y_neurons))):
        # odd rows will be offset to keep the hexagonal shape
        if y % 2 == 0:
            curRow = x * 3
        else:
            curRow = (x * 3) + 1
            
        curCol = (totCols - 2) - (y * 2)

        ax = plt.subplot2grid((totCols, totRows), (curCol, curRow), 
                              rowspan=2, colspan=2, projection=projection)
        ax.set_title((x, y))
        axs[(x, y)] = ax
        
    return fig, axs

def hex_heatmap(som, data, cmap='Blues', title='', cbLabel=''):
    # set up the figure
    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(111)
    ax.set_aspect('equal')
    cmap = mpl.colormaps[cmap]
    
    # get data from the som
    weights = som.get_weights()
    xx, yy = som.get_euclidean_coordinates()
    
    maxCount = max(v for v in data.values())
    minCount = min(v for v in data.values())
    
    # loops through the neurons
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            # Only use non-masked nodes
            if som._mask[i, j] == 0:
                # If theres no data still plot the hexagon
                if (i, j) not in data:
                    data[(i,j)] = 0
                    
                # determine the hexagon position and color
                wy = yy[(j, i)] * np.sqrt(3) / 2
                colorWeight = data[(i, j)]/maxCount
                
                # Create hexagon and add it to axis
                hex = patches.RegularPolygon((xx[(j, i)], wy), numVertices=6, 
                                             radius=.85 / np.sqrt(3), 
                                        facecolor=cmap(colorWeight), edgecolor='grey')
                ax.add_patch(hex)
                
                # determine the color the text should be based on color of node
                if colorWeight >= .75:
                    textColor = 'white'
                else:
                    textColor = 'black'
                
                # add text to hexagon for its frequency
                plt.text(xx[(j, i)], wy, f'{i}, {j}: {data[(i, j)]}', 
                         {'horizontalalignment': 'center', 'color': textColor})
            
    # align figure to show all hexagons
    plt.xlim(-1, weights.shape[0] - .5)
    plt.ylim(-1, (weights.shape[1] - .5) * np.sqrt(3) / 2)
    
    # remove the axis labels and lines
    ax.axis('off')
    
    # Create the color bar
    norm = mpl.colors.Normalize(vmin=minCount, vmax=maxCount)
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=.7)
    cb.set_label(cbLabel)
    
    # Title the plot
    plt.title(title, fontsize=20)
    
    return fig


def hex_frequency_plot(som, winmap=None):

    if winmap == None:
        winmap = som.win_map(dataarray)
    
    data = {k: len(v) for k, v in winmap.items()}
    
    fig = hex_heatmap(som, data, 'Blues', 'SOM Node Frequencies', 'Count')

    return fig

def hex_composite_map(som, winmap=None, somAvgs=None):
    #n = som._num
    #xy = som.xy_using_n(n)
    xy = 5
    fig, axs = hex_plot(xy, xy, projection=ccrs.PlateCarree())

    lats = dataset.sel(lat=latSlice, lon=lonSlice).lat
    lons = dataset.sel(lat=latSlice, lon=lonSlice).lon
    lons, lats = np.meshgrid(lons, lats)
    
    if winmap == None:
        winmap = som.win_map(dataarray, return_indices=True)
    neurons = list(winmap.keys())
            
    w = som._weights
    

    for neuron in neurons:
        avgs = np.array(w[neuron[0], neuron[1], :]).reshape((lons.shape[0], lats.shape[1]))

        axs[neuron[1], neuron[0]].pcolor(
            lons, lats, avgs, cmap='seismic', shading='nearest', transform=ccrs.PlateCarree())
        
        axs[neuron[1], neuron[0]].set_title(
            f"Sample size: {len(winmap[(neuron[0], neuron[1])])}: {(neuron[0], neuron[1])})", fontsize=12)
        
        # background map features
        axs[neuron[1], neuron[0]].coastlines(resolution='110m', color='k', linewidth=0.75, zorder=10)
        axs[neuron[1], neuron[0]].margins(x=0, y=0)
        axs[neuron[1], neuron[0]].add_feature(
            cfeature.STATES, facecolor='none', edgecolor='k', linewidth=0.35, zorder=10)
        axs[neuron[1], neuron[0]].add_feature(cf.BORDERS, linewidth=0.35, zorder=10)

    return fig

### Training

In [ ]:
inputLength = 
n = 4
sigma = 4
learning_rate = .001
neighborhood_function = 'gaussian''
decay_function = 'asymptotic_decay'
random_seed = 28
topology = 'hexagonal'
iteration = 100

# TRAINING
som = hexMinisom.MiniSom(inputLength, num=n, sigma=sigma, learning_rate=learning_rate, 
            neighborhood_function=neighborhood_function, decay_function=decay_function, 
            random_seed=random_seed, topology=topology)
som.pca_weights_init(dataarray)
som.train(dataarray, iteration, random_order=True, use_epochs=True)

# calculate multiple different errors
terror = som.topographic_error(dataarray)
perror = pathway_error(som, dataarray)

fig = hex_frequency_plot(som)
fileName = str(sigma) + '_' + str(learning_rate) + '_' + str(iteration) + '.png'
plt.savefig('tuning_output/pca_weights/frequencies/' + fileName, bbox_inches='tight')

fig = hex_composite_map(som)
plt.savefig('tuning_output/pca_weights/composites/' + fileName, bbox_inches='tight')

In [ ]:
winmap = som.win_map(dataarray, return_indices=True)

w = som._weights
minimum_weight = -np.max(np.abs(w))
maximum_weight = np.max(np.abs(w))

# Calculate the node number for each coordinate
mask = som._mask
node_nums = {}
n = 1
for i in range(mask.shape[0])[::-1]:
    for j in range(mask.shape[1]):
        # only use non masked nodes
        if som._mask[i, j] == 0:
            node_nums[(i, j)] = n
            n += 1


In [ ]:
fig, axs = hex_plot(som, projection=ccrs.PlateCarree())

lats = dataset.sel(lat=latSlice, lon=lonSlice).lat
lons = dataset.sel(lat=latSlice, lon=lonSlice).lon - 360
lons, lats = np.meshgrid(lons, lats)

neurons = list(winmap.keys())

for neuron in neurons:
    avgs = np.array(w[neuron[0], neuron[1], :]).reshape((lons.shape[0], lats.shape[1]))
    axs[neuron[0], neuron[1]].pcolor(
        lons, lats, avgs, vmin=minimum_weight, vmax=maximum_weight, cmap='seismic', 
        shading='nearest', transform=ccrs.PlateCarree())
    axs[neuron[0], neuron[1]].set_title(f"{node_nums[neuron]}", fontsize=16)
    
    # background map features
    axs[neuron[0], neuron[1]].coastlines(resolution='110m', color='k', linewidth=0.5, zorder=10)
    axs[neuron[0], neuron[1]].margins(x=0, y=0)
    axs[neuron[0], neuron[1]].add_feature(
        cfeature.STATES, facecolor='none', edgecolor='grey', linewidth=0.25, zorder=10
    )
    axs[neuron[0], neuron[1]].add_feature(cf.BORDERS, linewidth=0.25, zorder=10, edgecolor='grey')

cbar_ax = fig.add_axes([0.349, 0.165, 0.4, 0.015])
cmap = mpl.cm.seismic
norm = mpl.colors.Normalize(vmin=minimum_weight, vmax=maximum_weight)
cb = mpl.colorbar.ColorbarBase(ax=cbar_ax, cmap=cmap, norm=norm, orientation='horizontal', extend='both')
cb.ax.tick_params(labelsize=16)
cb.set_label('Z500 (\u03C3)', loc='center', fontsize=16)

plt.subplots_adjust(wspace=-0.475, hspace=-0.75)
plt.savefig('composites.pdf', transparent=True, bbox_inches='tight')